# MedQ / AI Core — EDA guiada + introdução aos modelos de Knowledge Tracing

Este notebook foi pensado para servir a dois objetivos ao mesmo tempo:

1. **Analisar os dados passo a passo**, com explicações claras e visuais que possam ser reutilizados em apresentação.
2. **Introduzir os modelos do projeto** — DKT, LPKT e DKVMN — explicando o papel de cada um, suas vantagens e quando cada abordagem tende a funcionar melhor.

---

## Estrutura do notebook

1. Contexto do problema
2. O que é Knowledge Tracing e por que ele importa
3. Visão geral dos modelos do projeto
4. Carregamento dos dados
5. Qualidade e cobertura da base
6. Sequências dos alunos
7. Tempo, sessões e fragmentação
8. Repetição por skill e implicações pedagógicas
9. Conclusões acionáveis para modelagem
10. Próximos passos


## 1. Contexto do problema

O objetivo central deste projeto é modelar, a partir das interações dos alunos com questões, uma estimativa dinâmica do seu estado de conhecimento.

Em termos práticos, queremos responder perguntas como:

- Qual a probabilidade de o aluno acertar a próxima questão?
- Quais conceitos parecem mais frágeis?
- O aluno está seguindo um padrão consistente de aprendizagem ou um comportamento fragmentado?
- Quais modelos capturam melhor essa dinâmica?


## 2. O que é Knowledge Tracing?

**Knowledge Tracing (KT)** é a tarefa de estimar o estado de conhecimento de um aluno ao longo do tempo com base em sua sequência de interações.

A ideia geral é simples:

- o aluno responde questões
- cada resposta traz evidências sobre o que ele sabe ou não sabe
- o modelo atualiza sua crença sobre o conhecimento do aluno
- essa crença pode ser usada para previsão e recomendação


## 3. Modelos do projeto: visão executiva

### 3.1 DKT — Deep Knowledge Tracing

O DKT modela a sequência do aluno usando uma arquitetura recorrente.

**Vantagens:**
- baseline forte e simples
- robusto quando os dados são mais bagunçados
- funciona bem para previsão **next-item**

**Limites:**
- representa menos explicitamente a estrutura conceitual
- pode sofrer overfitting cedo

---

### 3.2 LPKT — Learning Process-aware Knowledge Tracing

O LPKT tenta modelar não apenas a sequência de respostas, mas também o **processo de aprendizagem**, usando informações de tempo e dinâmica temporal.

**Vantagens:**
- aproveita melhor sinais temporais
- tende a capturar melhor ganho e esquecimento quando a sequência é coerente

**Limites:**
- depende mais da qualidade do sinal temporal
- sofre quando os alunos estudam de forma muito fragmentada

---

### 3.3 DKVMN — Dynamic Key-Value Memory Network

O DKVMN introduz uma memória explícita de conceitos. Em vez de depender apenas da ordem da sequência, ele tenta manter um estado de memória associado a componentes conceituais do conhecimento.

**Vantagens:**
- combina muito bem com Q-Matrix e árvore de assuntos
- mais alinhado com a ideia de mastery por conceito
- promissor para casos em que há estrutura conceitual forte

**Limites:**
- mais pesado computacionalmente
- mais sensível a escolhas de janela, memória e implementação


## 4. Features que queremos apresentar

Ao longo do projeto, as features centrais são:

- `user_id`: identifica o aluno
- `question_id`: identifica a questão
- `skill_id`: skill / conceito base da questão
- `correct`: acerto ou erro
- `timestamp`: momento da interação
- `time_response`: tempo gasto na resposta
- mapeamentos `skill -> H2 / H3`: agregações conceituais mais altas

Essas features nos permitem modelar:

- comportamento sequencial
- tempo entre interações
- repetição por skill
- progressão conceitual em diferentes níveis


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

ANSWERS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset' / 'answers_prepared.csv'
SEQUENCES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sequences' / 'user_sequences.json'
H2_MAP_PATH = PROJECT_ROOT / 'data' / 'processed' / 'mappings' / 'skill_to_h2.json'
H3_MAP_PATH = PROJECT_ROOT / 'data' / 'processed' / 'mappings' / 'skill_to_h3.json'

assert ANSWERS_PATH.exists(), f'Arquivo não encontrado: {ANSWERS_PATH}'
assert SEQUENCES_PATH.exists(), f'Arquivo não encontrado: {SEQUENCES_PATH}'

answers = pd.read_csv(ANSWERS_PATH)
with open(SEQUENCES_PATH, 'r', encoding='utf-8') as f:
    sequences = json.load(f)

skill_to_h2 = {}
skill_to_h3 = {}
if H2_MAP_PATH.exists():
    with open(H2_MAP_PATH, 'r', encoding='utf-8') as f:
        skill_to_h2 = json.load(f)
if H3_MAP_PATH.exists():
    with open(H3_MAP_PATH, 'r', encoding='utf-8') as f:
        skill_to_h3 = json.load(f)

answers['timestamp_dt'] = pd.to_datetime(answers['timestamp'], unit='ms', errors='coerce')
answers.head()


## 5. Visão geral da base

Antes de pensar em modelos, precisamos validar se a base está saudável.

As perguntas aqui são:

- Qual o volume total de interações?
- Quantos alunos, questões e skills temos?
- Há duplicados ou nulos críticos?
- Qual a taxa global de acerto?


In [ ]:
overview = pd.Series({
    'n_rows': len(answers),
    'n_users': answers['user_id'].nunique(),
    'n_questions': answers['question_id'].nunique(),
    'n_skills': answers['skill_id'].nunique(),
    'correct_rate': float(answers['correct'].mean()),
    'duplicated_rows': int(answers.duplicated().sum()),
    'null_user_id': int(answers['user_id'].isna().sum()),
    'null_question_id': int(answers['question_id'].isna().sum()),
    'null_skill_id': int(answers['skill_id'].isna().sum()),
    'time_response_zero_rate': float((answers['time_response'] == 0).mean()),
})
overview.to_frame('value')


### Leitura para apresentação

Esta tabela serve para mostrar que o projeto parte de uma base estruturada e monitorada.

Pontos que normalmente merecem destaque:
- volume total de interações
- número de usuários distintos
- taxa global de acerto
- presença ou ausência de problemas críticos de qualidade


## 6. Distribuição de interações por aluno

Modelos de KT dependem da existência de histórico suficiente por aluno.

Nesta seção queremos mostrar:
- quantas interações cada aluno tem
- se a distribuição é muito desigual
- se o projeto sofre com sequências curtas


In [ ]:
user_stats = (
    answers.groupby('user_id')
    .agg(
        n_interactions=('question_id', 'size'),
        n_skills=('skill_id', 'nunique'),
        first_ts=('timestamp_dt', 'min'),
        last_ts=('timestamp_dt', 'max'),
    )
    .reset_index()
)
user_stats['timespan_hours'] = ((user_stats['last_ts'] - user_stats['first_ts']).dt.total_seconds() / 3600).fillna(0)
user_stats[['n_interactions', 'n_skills', 'timespan_hours']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.hist(user_stats['n_interactions'], bins=50)
plt.title('Distribuição de interações por aluno')
plt.xlabel('Interações por aluno')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()


### Interpretação

Este gráfico ajuda a discutir um ponto importante do projeto:

- se os históricos forem curtos, todos os modelos sofrem
- DKT costuma ser mais robusto
- LPKT e DKVMN tendem a ganhar mais quando existe sequência rica e informativa


## 7. Tempo entre interações e tempo de resposta

Aqui investigamos o papel do tempo na base.

Isso é especialmente importante porque:
- o LPKT depende mais do componente temporal
- ruído temporal reduz o ganho do modelo
- outliers podem contaminar a interpretação do comportamento do aluno


In [ ]:
answers = answers.sort_values(['user_id', 'timestamp']).copy()
answers['delta_t_ms'] = answers.groupby('user_id')['timestamp'].diff().fillna(0)
answers['delta_t_min'] = answers['delta_t_ms'] / 60000

time_summary = pd.Series({
    'delta_p50_min': float(answers['delta_t_min'].quantile(0.50)),
    'delta_p90_min': float(answers['delta_t_min'].quantile(0.90)),
    'delta_p99_min': float(answers['delta_t_min'].quantile(0.99)),
    'time_response_p50_sec': float(answers['time_response'].quantile(0.50) / 1000),
    'time_response_p90_sec': float(answers['time_response'].quantile(0.90) / 1000),
    'time_response_p99_sec': float(answers['time_response'].quantile(0.99) / 1000),
    'time_response_zero_rate': float((answers['time_response'] == 0).mean()),
})
time_summary.to_frame('value')


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.hist(np.log1p(answers['delta_t_ms']), bins=60)
plt.title('Distribuição de log1p(delta_t_ms)')
plt.xlabel('log1p(delta_t_ms)')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()

fig = plt.figure(figsize=(10, 5))
answers['time_response'].clip(upper=answers['time_response'].quantile(0.99)).hist(bins=60)
plt.title('Distribuição de time_response (capado no p99)')
plt.xlabel('time_response')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()


### Interpretação para apresentação

Este bloco é ideal para justificar por que o tempo é uma feature útil, mas também imperfeita.

Mensagens que podem ser destacadas:
- o componente temporal existe e carrega sinal
- porém a distribuição costuma ser muito assimétrica
- isso ajuda a explicar por que o LPKT depende tanto da qualidade da sequência


## 8. Sessões de estudo

Em vez de assumir que todo o histórico do aluno é uma única trilha contínua, podemos agrupá-lo em sessões.

Isso é importante porque:
- sessões curtas indicam estudo fragmentado
- sequências por sessão podem melhorar a qualidade do input para o KT
- essa análise ajuda a explicar decisões recentes do projeto


In [ ]:
SESSION_GAP_MIN = 60
answers['new_session_flag'] = ((answers['delta_t_min'] > SESSION_GAP_MIN) | (answers.groupby('user_id').cumcount() == 0)).astype(int)
answers['session_idx'] = answers.groupby('user_id')['new_session_flag'].cumsum()

session_stats = (
    answers.groupby(['user_id', 'session_idx'])
    .agg(
        n_interactions=('question_id', 'size'),
        n_skills=('skill_id', 'nunique'),
        correct_rate=('correct', 'mean'),
    )
    .reset_index()
)
session_stats[['n_interactions', 'n_skills', 'correct_rate']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.hist(session_stats['n_interactions'], bins=40)
plt.title('Distribuição do tamanho das sessões')
plt.xlabel('Interações por sessão')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()


### Interpretação

Esta análise é especialmente útil para justificar:
- o uso de sequências por sessão
- por que LPKT e DKVMN podem depender de entradas mais limpas
- por que o histórico global do aluno nem sempre representa bem o processo de aprendizagem


## 9. Fragmentação do estudo

Agora queremos medir se o aluno aprofunda uma skill por vez ou se troca de skill o tempo todo.

Isso impacta diretamente a interpretação dos modelos:
- DKT tolera mais fragmentação
- LPKT sofre se o processo não for coerente
- DKVMN pode compensar parte disso quando a estrutura conceitual é forte


In [ ]:
answers['prev_skill_id'] = answers.groupby('user_id')['skill_id'].shift(1)
answers['skill_switch'] = ((answers['prev_skill_id'].notna()) & (answers['skill_id'] != answers['prev_skill_id'])).astype(int)

frag = (
    answers.groupby('user_id')
    .agg(
        n_interactions=('question_id', 'size'),
        n_skills=('skill_id', 'nunique'),
        switches=('skill_switch', 'sum'),
    )
    .reset_index()
)
frag['switch_rate'] = frag['switches'] / (frag['n_interactions'] - 1).clip(lower=1)
frag['interactions_per_skill'] = frag['n_interactions'] / frag['n_skills'].clip(lower=1)
frag[['switch_rate', 'interactions_per_skill']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.hist(frag['switch_rate'], bins=40)
plt.title('Taxa de troca de skill por aluno')
plt.xlabel('Skill switch rate')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()

fig = plt.figure(figsize=(10, 5))
plt.hist(frag['interactions_per_skill'], bins=40)
plt.title('Interações por skill por aluno')
plt.xlabel('Interações / skill')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()


## 10. Repetição por skill

Repetição é um elemento importante para que o modelo perceba ganho, esquecimento e recuperação de conhecimento.

Aqui medimos quantas vezes uma combinação `aluno × skill` aparece na base.


In [ ]:
user_skill_counts = answers.groupby(['user_id', 'skill_id']).size().reset_index(name='n_attempts')

repeat_summary = pd.Series({
    'mean_attempts_per_user_skill': float(user_skill_counts['n_attempts'].mean()),
    'median_attempts_per_user_skill': float(user_skill_counts['n_attempts'].median()),
    'share_attempted_once': float((user_skill_counts['n_attempts'] == 1).mean()),
    'share_attempted_ge_3': float((user_skill_counts['n_attempts'] >= 3).mean()),
    'share_attempted_ge_5': float((user_skill_counts['n_attempts'] >= 5).mean()),
})
repeat_summary.to_frame('value')


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.hist(user_skill_counts['n_attempts'].clip(upper=user_skill_counts['n_attempts'].quantile(0.99)), bins=40)
plt.title('Distribuição de tentativas por combinação aluno × skill')
plt.xlabel('Número de tentativas')
plt.ylabel('Frequência')
plt.grid(alpha=0.25)
plt.show()


## 11. Cobertura conceitual H2 / H3

Como parte do projeto, usamos mapeamentos de `skill -> H2 / H3`.

Esta etapa ajuda a responder:
- quão bem a estrutura conceitual cobre as skills observadas
- até que ponto os experimentos H2/H3 representam o comportamento real da base


In [ ]:
skills_observed = set(answers['skill_id'].dropna().astype(str).unique())
coverage_h2 = len(skills_observed & set(skill_to_h2.keys())) / max(len(skills_observed), 1) if skill_to_h2 else np.nan
coverage_h3 = len(skills_observed & set(skill_to_h3.keys())) / max(len(skills_observed), 1) if skill_to_h3 else np.nan

coverage = pd.Series({
    'observed_skills': len(skills_observed),
    'mapped_h2_skills': len(skills_observed & set(skill_to_h2.keys())) if skill_to_h2 else np.nan,
    'mapped_h3_skills': len(skills_observed & set(skill_to_h3.keys())) if skill_to_h3 else np.nan,
    'coverage_h2': coverage_h2,
    'coverage_h3': coverage_h3,
})
coverage.to_frame('value')


## 12. Conclusões acionáveis para modelagem

A partir desta EDA, as leituras esperadas para discussão são:

### Quando DKT tende a ir bem
- quando a sequência existe, mas o comportamento do aluno é mais irregular
- quando precisamos de um baseline forte e estável

### Quando LPKT tende a ir bem
- quando o sinal temporal é informativo
- quando as sessões são mais coerentes
- quando o processo de estudo se aproxima de uma trajetória de aprendizagem mais limpa

### Quando DKVMN tende a ir bem
- quando temos uma boa estrutura conceitual
- quando queremos modelar memória por conceito
- quando Q-Matrix / árvore de assuntos fazem parte central do projeto


## 13. Próximos passos

Este notebook prepara o terreno para a etapa seguinte do projeto:

1. comparar os modelos DKT, LPKT e DKVMN
2. analisar **next-item**, **H2** e **H3**
3. discutir trade-offs entre robustez sequencial, sinal temporal e memória conceitual
4. transformar os melhores resultados em benchmark oficial do projeto
